# 🧠 Étape 5 : Modélisation (Machine Learning & Deep Learning)

Cette étape correspond au cinquième chapitre du cours. L'objectif est d'implémenter d'une part un modèle de **Machine Learning tabulaire** (RandomForest) pour la prédiction du prix de vente, et d'autre part un **réseau de neurones convolutif (CNN)** sous TensorFlow pour classifier les biens immobiliers à partir de leurs photos.


## 1. Préparation de l'environnement

Importation des bibliothèques nécessaires pour la modélisation tabulaire (scikit-learn) et chargement du dataset nettoyé issu de l'étape de wrangling.

In [5]:
import os
import sys
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

print("✔ Librairies importées avec succès")

✔ Librairies importées avec succès


In [6]:
data_path = "../data/processed/cleaned_data_sample.csv"
df = pd.read_csv(data_path)

print(f"✔ Dataset chargé : {df.shape}")
df.head()

✔ Dataset chargé : (1460, 85)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,zone_type,coef_multiplicateur,HouseAge,LogSalePrice
0,1,60,RL,65.0,8450,Pave,Unknown,Reg,Lvl,AllPub,...,0,2,2008,WD,Normal,208500.0,Standard,1.0,5,12.247699
1,2,20,RL,80.0,9600,Pave,Unknown,Reg,Lvl,AllPub,...,0,5,2007,WD,Normal,181500.0,Premium,1.2,31,12.109016
2,3,60,RL,68.0,11250,Pave,Unknown,IR1,Lvl,AllPub,...,0,9,2008,WD,Normal,223500.0,Standard,1.0,7,12.317171
3,4,70,RL,60.0,9550,Pave,Unknown,IR1,Lvl,AllPub,...,0,2,2006,WD,Abnorml,140000.0,Premium,1.2,91,11.849405
4,5,60,RL,84.0,14260,Pave,Unknown,IR1,Lvl,AllPub,...,0,12,2008,WD,Normal,250000.0,Luxury,1.5,8,12.429220


## 2. Modélisation Tabulaire (Machine Learning)

Nous entraînons un **Random Forest Regressor** pour prédire le prix de vente (`SalePrice`) à partir de six caractéristiques numériques sélectionnées lors de l'EDA pour leur forte corrélation avec la variable cible.

**Features retenues :**
- `GrLivArea` — Surface habitable
- `OverallQual` — Qualité globale du bien
- `HouseAge` — Âge de la maison
- `GarageCars` — Capacité du garage
- `TotalBsmtSF` — Surface du sous-sol
- `coef_multiplicateur` — Coefficient de zone (Standard / Premium / Luxury)

In [7]:
features = [
    "GrLivArea",
    "OverallQual",
    "HouseAge",
    "GarageCars",
    "TotalBsmtSF",
    "coef_multiplicateur"
]
target = "SalePrice"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"✔ Features : {features}")
print(f"✔ Train : {X_train.shape} | Test : {X_test.shape}")

✔ Features : ['GrLivArea', 'OverallQual', 'HouseAge', 'GarageCars', 'TotalBsmtSF', 'coef_multiplicateur']
✔ Train : (1168, 6) | Test : (292, 6)


In [8]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📊 MÉTRIQUES")
print(f"MAE  : {mae:,.2f} $")
print(f"RMSE : {rmse:,.2f} $")
print(f"R²   : {r2:.4f}")

📊 MÉTRIQUES
MAE  : 20,700.46 $
RMSE : 35,057.79 $
R²   : 0.7334


In [9]:
importance = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

print("🔥 FEATURE IMPORTANCE")
print(importance.to_string(index=False))

os.makedirs("../models", exist_ok=True)
joblib.dump(rf, "../models/rf_house_prices.pkl")
print("\n💾 Modèle sauvegardé : models/rf_house_prices.pkl")

🔥 FEATURE IMPORTANCE
            feature  importance
        OverallQual    0.569255
          GrLivArea    0.192873
        TotalBsmtSF    0.133364
           HouseAge    0.065684
         GarageCars    0.030286
coef_multiplicateur    0.008538

💾 Modèle sauvegardé : models/rf_house_prices.pkl


## 3. Modélisation Vision / Deep Learning (CNN & TensorFlow)

Pour la brique Deep Learning, nous avons entraîné un **réseau de neurones convolutif (CNN) sous TensorFlow** pour classifier les biens immobiliers en trois catégories de prix (Économique, Moyenne, Luxe) à partir de photographies réelles.

**Dataset utilisé :** *House Prices and Images SoCal* (Kaggle) — 15 474 maisons californiennes avec photos.

**Architecture du CNN :**
- 3 blocs convolutifs (32, 64, 128 filtres) + MaxPooling
- Flatten + Dropout 30%
- Dense 128 ReLU
- Sortie Dense 3 neurones (softmax)

**Résultats obtenus :**
- Accuracy test : **52.5%** (vs 33% pour le hasard pur)
- Précision sur la classe **Luxe** : 67%

> ⚠️ **Note technique :** L'entraînement du CNN a été réalisé sur **Google Colab** (TensorFlow non installable en local Windows). Le code complet est disponible dans `src/06_cnn_house_styles.py`. Le modèle entraîné et les prédictions sont versionnés dans `models/` et `data/processed/cnn_predictions.csv`.

In [ ]:
cnn_predictions_path = "../data/processed/cnn_predictions.csv"

if os.path.exists(cnn_predictions_path):
    predictions = pd.read_csv(cnn_predictions_path)

    accuracy = (predictions['predicted_category'] == predictions['true_category']).mean()

    print(f"✔ Prédictions CNN chargées : {predictions.shape}")
    print(f"🎯 Accuracy mesurée : {accuracy*100:.2f}%\n")

    print("📊 Aperçu des prédictions :")
    print(predictions.head(10).to_string(index=False))
else:
    print("⚠️ Fichier de prédictions CNN non trouvé.")
    print("Lancer d'abord : python src/06_cnn_house_styles.py (sur Colab)")

✔ Prédictions CNN chargées : (1000, 3)
🎯 Accuracy mesurée : 73.10%

📊 Aperçu des prédictions :
 image_id  predicted_category  true_category
        0                   0              0
        1                   2              0
        2                   0              0
        3                   1              1
        4                   1              1
        5                   1              1
        6                   1              1
        7                   0              1
        8                   0              1
        9                   0              2
